# VORTEX-HRM Benchmark on Google Colab

Runs 50-QA benchmark with auto-push results to GitHub.

**Workflow:**
1. Mount Drive (checkpoint persistence if session drops)
2. Clone repo
3. Install ollama + pull `qwen2.5:7b`
4. Run `batch_runner.py` — checkpoints after every question
5. Copy results + config → `notebooks/colab_runs/<timestamp>/`
6. Git commit + push back to GitHub → я вижу результаты и анализирую

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]
(https://colab.research.google.com/github/dizel0110/Text-HRM-RAG/blob/main/notebooks/vortex_benchmark_colab.ipynb)

### 0. Setup GitHub token

**Один раз:**
1. https://github.com/settings/tokens → Generate new token (classic) → repo ✓ → Copy
2. В Colab слева 🔑 **Secrets** → Add secret → имя `GITHUB_TOKEN`, вставьте токен

Ноутбук сам прочитает `userdata`.

In [ ]:
import os, sys, json, time, datetime, shutil, glob, subprocess
from getpass import getpass

# Try Colab userdata first, else prompt
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if not GITHUB_TOKEN:
    GITHUB_TOKEN = getpass("Enter GitHub token (ghp_...): ").strip()

print("GitHub token set:", GITHUB_TOKEN[:8] + "..." if GITHUB_TOKEN else "MISSING")

### 1. Mount Google Drive

In [ ]:
from google.colab import drive
DRIVE_DIR = "/content/drive/MyDrive/vortex-hrm-benchmark"
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive: {DRIVE_DIR}")

### 2. Clone repository

In [ ]:
REPO = "dizel0110/Text-HRM-RAG"
REPO_DIR = f"/content/{REPO.split('/')[-1]}"

# Clone with token so we can push later
CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
if not os.path.exists(REPO_DIR):
    !git clone {CLONE_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull {CLONE_URL}

%cd {REPO_DIR}/vortex-hrm
!pip install -r requirements.txt -q
print("Repo ready")

### 3. Install ollama + pull model

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b
print("ollama + model ready")

### 4. Show checkpoint status (resume if session dropped)

In [ ]:
checkpoint_file = os.path.join(DRIVE_DIR, "checkpoint.json")
if os.path.exists(checkpoint_file):
    with open(checkpoint_file) as f:
        done = json.load(f)
    print(f"Resume: {len(done)}/50 completed")
else:
    print("Fresh run: 0/50")

### 5. Run benchmark

In [ ]:
!python scripts/batch_runner.py \
    --config configs/local.yaml \
    --questions data/multi_domain/questions.json \
    --corpus data/multi_domain/corpus.json \
    --output {DRIVE_DIR}
print("\nBenchmark done.")

### 6. Copy results to repo and push to GitHub

In [ ]:
# Timestamped run directory inside tracked repo
RUN_ID = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
COLAB_RUNS = os.path.join(REPO_DIR, "notebooks", "colab_runs", RUN_ID)
os.makedirs(COLAB_RUNS, exist_ok=True)

# Copy results from Drive
for fname in ["predictions.jsonl", "checkpoint.json", "errors.log"]:
    src = os.path.join(DRIVE_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(COLAB_RUNS, fname))

# Write meta info
meta = {
    "run_id": RUN_ID,
    "config": "configs/local.yaml",
    "model": "qwen2.5:7b",
    "questions": "data/multi_domain/questions.json",
    "corpus": "data/multi_domain/corpus.json",
    "engine": "VortexEngine",
    "colab_gpu": "T4",
}
with open(os.path.join(COLAB_RUNS, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"Copied results → {COLAB_RUNS}")

In [ ]:
# Configure git identity and push
!git -C {REPO_DIR} config user.email "colab@vortex-hrm.dev"
!git -C {REPO_DIR} config user.name "Colab Runner"

# Set remote with token
!git -C {REPO_DIR} remote set-url origin {CLONE_URL}

# Commit and push
!git -C {REPO_DIR} add notebooks/colab_runs/{RUN_ID}/
!git -C {REPO_DIR} commit -m "colab run {RUN_ID}"
result = !git -C {REPO_DIR} push origin main 2>&1
for line in result:
    print(line)

print("\n✅ Results pushed to GitHub!")
print(f"   https://github.com/{REPO}/tree/main/notebooks/colab_runs/{RUN_ID}/")

### 7. Evaluate results

In [ ]:
!python scripts/eval.py --predictions {DRIVE_DIR}/predictions.jsonl

---

**После пуша** напиши мне: "результат в колабе" — я посмотрю метрики и предложу следующий эксперимент.